# Evaluating LLM Outputs: With an Answer Key vs Without

In normal software, a test passes or fails. In LLM systems, **"is this a good answer?" is not binary**. An answer can be accurate but vague, fluent but wrong in one number, or well written but not grounded in the evidence.

So how do we score answers? Two families of methods:

| Method | Idea | We will see |
|---|---|---|
| **Reference-based** | Compare the answer to a ground-truth answer (an *answer key*) | Where it works, and where it breaks |
| **Reference-free** | Judge the answer on its own properties, no answer key | How an **LLM judge** does it, its 4 modes, and its 5 known biases |

Everything here uses **real data** (SQuAD and Dolly from Hugging Face) and **real LLM calls** (Groq). No fake outputs, no fake scores.

| Section | What happens |
|---|---|
| 0 | Setup: a model to evaluate and a model to judge |
| 1 | Why a normal `assert` test fails on LLM output |
| 2 | Reference-based evaluation |
| 3 | Reference-free evaluation |
| 4 | LLM as judge: the four modes |
| 5 | Judge biases |
| 6 | Final comparison |

> Samples are small (6 to 8 examples per experiment) so the notebook runs on the free tier. The numbers **illustrate each mechanism**, they are not statistics. LLM outputs also vary a little between runs, and Groq's free tier may ask you to wait: just re-run the cell.

# 0. Setup

Two models, two roles:
- **Generator** = the model whose answers we evaluate.
- **Judge** = a model from a **different family**, so it doesn't favour its own style (one of the biases we meet in Section 5). Ideally the judge is at least as strong as the generator; check the model sizes you choose.

In [10]:
%pip install -q langchain-groq datasets

In [11]:
import re
import json
import pandas as pd
from collections import Counter
from google.colab import userdata
from datasets import load_dataset
from langchain_groq import ChatGroq

# Bring the api_key from secrets into a variable
my_api_key = userdata.get('agentic_ai')

# Show full text inside tables
pd.set_option('display.max_colwidth', 90)

In [12]:
# The model we are going to EVALUATE
# Groq's free tier has a separate daily token limit for EACH model. If you hit a
# "tokens per day" (429) error on this model, switch to another model you have access to.
GENERATOR_MODEL = 'openai/gpt-oss-120b'

generator = ChatGroq(
    model=GENERATOR_MODEL,
    temperature=0,
    api_key=my_api_key,
    max_retries=6   # retry automatically if Groq's free tier says "slow down"
)

# The model that will JUDGE (different family from the generator)
# If this model is not available for you, pick another model from a DIFFERENT family
# than the generator (a same-family judge brings in "self-preference", see Section 5)
JUDGE_MODEL = 'qwen/qwen3.8-27b'

judge = ChatGroq(
    model=JUDGE_MODEL,
    temperature=0,
    api_key=my_api_key,
    max_retries=6
)

## 0.1 Two small helpers

- `ask(llm, prompt)` sends a prompt and gives back the text. If the model returns a blank answer it retries, and stops with an error if it is still blank, so an empty answer is never scored as a real one.
- `ask_judge(prompt)` asks the judge to reply in **JSON** (so we can read scores in code) and turns the reply into a Python dictionary.

In [13]:
def ask(llm, prompt):
    """Send one prompt to a model and return its text answer (never empty)."""
    text = llm.invoke(prompt).content
    for _ in range(2):
        if text.strip():
            break
        # Reasoning models sometimes return no visible text. Ask again and nudge for a visible answer.
        text = llm.invoke(prompt + "\n\nReply with a short visible answer.").content
    if not text.strip():
        # Stop here: a blank answer scored as if it were a real answer would corrupt the results
        raise ValueError("The model returned an empty answer after retries.")
    return text

def ask_judge(prompt):
    """Ask the judge for a JSON reply and return it as a dictionary."""
    for attempt in range(3):
        text = judge.invoke(prompt).content
        # Some models print their thinking inside <think> tags. Remove it.
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
        try:
            # Keep only the part between the first { and the last }
            return json.loads(text[text.index("{"): text.rindex("}") + 1])
        except ValueError:
            continue   # not valid JSON, ask again
    raise ValueError("Judge did not return valid JSON: " + text)

In [14]:
# Quick check that both models are alive
print("Generator:", ask(generator, "Say hello in 5 words."))
print("Judge    :", ask_judge('Reply with JSON only: {"status": "ready"}'))

Generator: Hello there, nice to meet!
Judge    : {'status': 'ready'}


**What you're seeing:** a greeting from the generator, and a Python dictionary from the judge. If both print, we're ready.

# 1. Why Can't We Just Write a Test?

In normal software we write `assert output == expected`. Let's try that on an LLM with two real questions whose answers we know.

In [15]:
tests = [
    ("What is the capital of Australia?", "Canberra"),
    ("Who wrote the Indian national anthem?", "Rabindranath Tagore"),
]

for question, expected in tests:
    answer = ask(generator, question)
    print("Question :", question)
    print("Answer   :", answer)
    print("answer == expected ->", answer == expected, "\n")

Question : What is the capital of Australia?
Answer   : The capital of Australia is **Canberra**.
answer == expected -> False 

Question : Who wrote the Indian national anthem?
Answer   : The Indian national anthem, **“Jana Gana Mana,”** was written and composed by **Rabindranath Tagore**. He penned the original Bengali lyrics in 1911, and the same melody he set to those words is used for the anthem today.
answer == expected -> False 



**What you're seeing:** the model most likely answered **correctly**, but in a full sentence, so the exact-match test says `False`. A correct answer failed our test.

That one example is the whole problem: there are many correct ways to say the same thing. So how do we score answers properly? Let's start with the classic approach.

# 2. Reference-Based Evaluation: Grading with an Answer Key

Compare the model's answer with a **ground-truth answer**. It's like marking an exam with an answer key.

## 2.1 Where it works: extractive questions (one right answer)

We use **SQuAD**, a real reading-comprehension dataset. Each question has a passage and a short, exact answer taken from it.

In [16]:
# Load 8 random real questions from SQuAD
squad = load_dataset("rajpurkar/squad", split="validation").shuffle(seed=7).select(range(8))

example = squad[0]
print("Question    :", example["question"])
print("Gold answers:", example["answers"]["text"])

Question    : Who designed the ceiling and stained-glass windows of the Green Dining Room?
Gold answers: ['Edward Burne-Jones', 'Edward Burne-Jones', 'Edward Burne-Jones']


**What you're seeing:** a real question and its gold answer (SQuAD stores a few accepted spellings of the answer).

## 2.2 How do we score? Exact Match and F1

- **Exact Match (EM):** 1 if the answer equals a gold answer (after lowercasing and removing punctuation and articles), else 0.
- **F1:** word overlap between the answer and the gold answer. Partly right answers get partial credit.

In [17]:
def normalize(text):
    """Lowercase, remove punctuation and articles, split into words."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # remove punctuation
    text = re.sub(r"\b(a|an|the)\b", " ", text)   # remove a / an / the
    return text.split()

def exact_match(answer, golds):
    """1 if the answer equals any gold answer."""
    return float(any(normalize(answer) == normalize(g) for g in golds))

def f1(answer, golds):
    """Best word-overlap F1 against any gold answer."""
    best = 0.0
    for g in golds:
        answer_words, gold_words = normalize(answer), normalize(g)
        common = sum((Counter(answer_words) & Counter(gold_words)).values())
        if common == 0:
            continue
        precision = common / len(answer_words)
        recall = common / len(gold_words)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best

## 2.3 Run the generator and score it

We ask the generator to answer using only the passage, with just the answer words. Then we score every answer against the gold answer.

In [18]:
open_book_answers = []   # answers given WITH the passage

for row in squad:
    prompt = (
        f"Context: {row['context']}\n\n"
        f"Question: {row['question']}\n\n"
        "Answer using only the context. Reply with just the answer words, nothing else."
    )
    open_book_answers.append(ask(generator, prompt))

In [19]:
rows = []
for row, answer in zip(squad, open_book_answers):
    golds = row["answers"]["text"]
    rows.append({
        "question": row["question"],
        "gold": golds[0],
        "model answer": answer,
        "EM": exact_match(answer, golds),
        "F1": round(f1(answer, golds), 2),
    })

squad_df = pd.DataFrame(rows)
squad_df

,question,gold,model answer,EM,F1
0,Who designed the ceiling and stained-glass windows of the Green Dining Room?,Edward Burne-Jones,Edward Burne‑Jones,1.0,1.00
1,What was the name of the first Doctor Who radio drama?,Slipback,Slipback,1.0,1.00
2,Who compiled the original surviving Apollo 11 landing data?,Nafzger,Nafzger,1.0,1.00
3,What was the Doritos customer Super Bowl ad campaign called?,Crash the Super Bowl,Crash the Super Bowl,1.0,1.00
4,The price of oil is usually a stable commodity until when?,Until the oil shock,the oil shock,1.0,1.00
5,What church is located at the corner of 4th and New Streets in Philadelphia?,St. George's United Methodist Church,St. George's United Methodist Church,1.0,1.00
6,What did Paul Rose say Luther added to German thought?,hysterical and demonizing mentality,a hysterical and demonizing mentality about Jews,1.0,1.00
7,Who is in control of legislative power?,both the government and the National Assembly and the Senate,"the government, the National Assembly and the Senate",0.0,0.91


In [20]:
print("Average Exact Match:", squad_df["EM"].mean())
print("Average F1         :", round(squad_df["F1"].mean(), 2))

Average Exact Match: 0.875
Average F1         : 0.99


**What you're seeing:** the model's answer next to the gold answer, and a score per row. When the task has **one right answer**, reference-based scoring is cheap, objective and repeatable: no judge needed, just arithmetic.

## 2.4 Where it breaks: open-ended questions

Now a different kind of task. **Dolly** is a real dataset of questions with answers written by humans. There are many good ways to answer these, and the human reference is just one of them.

In [21]:
# Real open-ended questions with a human-written reference answer
dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

# Keep general questions whose reference is 20 to 80 words long, then pick 6
dolly = dolly.filter(lambda r: r["category"] == "general_qa" and 20 <= len(r["response"].split()) <= 80)
dolly = dolly.shuffle(seed=1).select(range(6))

questions = list(dolly["instruction"])
references = list(dolly["response"])

print("Question :", questions[0])
print("Reference:", references[0])

Question : What is the standard tuning for a 6 string guitar?
Reference: From the highest tone to the lowest, the tuning of the 6 strings is E B G D A E.


In [22]:
# Ask the generator each question, and keep the short answer
concise_answers = [ask(generator, q + "\n\nAnswer in 2 sentences.") for q in questions]

# Score against the human reference with the same F1 as before
dolly_df = pd.DataFrame({
    "question": [q[:70] for q in questions],
    "F1 vs reference": [round(f1(a, [r]), 2) for a, r in zip(concise_answers, references)],
})
dolly_df

,question,F1 vs reference
0,What is the standard tuning for a 6 string guitar?,0.54
1,What does a typical financial planning and analysis team structure con,0.23
2,How should I pick a name for my baby?,0.22
3,What is a jet engine?,0.29
4,How do I turn on my Tesla Y?,0.35
5,Do you prefer a Tesla model S or a Porsche Taycan,0.26


In [23]:
# Read two examples side by side
for i in range(2):
    print("Question :", questions[i])
    print("Reference:", references[i])
    print("Model    :", concise_answers[i])
    print("F1       :", dolly_df["F1 vs reference"][i], "\n")

Question : What is the standard tuning for a 6 string guitar?
Reference: From the highest tone to the lowest, the tuning of the 6 strings is E B G D A E.
Model    : The standard tuning for a six‑string guitar is E‑A‑D‑G‑B‑E, from the lowest (thickest) string to the highest (thinnest). This arrangement is often remembered as “EADGBE.”
F1       : 0.54 

Question : What does a typical financial planning and analysis team structure consist of?
Reference: Typically, a financial planning and analysis team will consist of a financial analyst, senior financial analyst, finance manager, director, and VP. This team will usually fall under the chief financial officer for the company.
Model    : A typical FP&A team is led by a VP or Director of FP&A, who oversees managers responsible for core functions such as budgeting, forecasting, and strategic analysis, each managing senior analysts and analysts who produce the financial models, reports, and insights. Supporting roles often include data‑engine

**What you're seeing:** F1 scores that are probably **low**, yet when you read the model's answer next to the reference, it is often a perfectly good answer. It just uses **different words**.

That is the weakness of reference-based scoring for open-ended text: **many answers are correct, and string overlap punishes valid paraphrases.** (In Section 4 we fix this with a judge.)

| | Reference-based |
|---|---|
| **Advantages** | Cheap, objective, repeatable. Perfect for tasks with one right answer (extraction, classification). |
| **Disadvantages** | Needs a ground-truth answer for every case. Punishes valid paraphrases. Measures wording, not meaning. |

In [24]:
# Keep the numbers for the final comparison
summary = []
summary.append(("Reference-based | SQuAD (one right answer) | average F1", round(squad_df["F1"].mean(), 2)))
summary.append(("Reference-based | Dolly (open-ended) | average F1", round(dolly_df["F1 vs reference"].mean(), 2)))

# 3. Reference-Free Evaluation: No Answer Key

Now the judge sees **only the question, the context and the answer**. It never sees the gold answer. It scores whether the answer is **faithful to the context**.

This scales: nobody has to write ground truth for every case. The price is trust: we must check whether the judge is reliable.

## 3.1 Make a mix of good and bad answers (for real)

To test a judge we need both good and bad answers. We get them honestly from the generator:
- **with the passage** (we already have these),
- **without the passage**, so the model has to answer from memory and will sometimes be vague or wrong.

In [25]:
closed_book_answers = []   # answers given WITHOUT the passage

for row in squad:
    closed_book_answers.append(ask(generator, row["question"] + " Answer in one short sentence."))

for row, answer in list(zip(squad, closed_book_answers))[:3]:
    print("Question:", row["question"])
    print("Without the passage:", answer)
    print("Gold:", row["answers"]["text"][0], "\n")

Question: Who designed the ceiling and stained-glass windows of the Green Dining Room?
Without the passage: Louis Comfort Tiffany designed the ceiling and stained‑glass windows of the Green Dining Room.
Gold: Edward Burne-Jones 

Question: What was the name of the first Doctor Who radio drama?
Without the passage: The first Doctor Who radio drama was “The Pescatons.”
Gold: Slipback 

Question: Who compiled the original surviving Apollo 11 landing data?
Without the passage: NASA’s Manned Spacecraft Center (now Johnson Space Center) compiled the original surviving Apollo 11 landing data.
Gold: Nafzger 



**What you're seeing:** some of these answers match the gold, and some don't. We have a natural mix.

## 3.2 The faithfulness judge

In [26]:
def faithfulness(context, question, answer):
    """Reference-free: is the answer supported by the context? Score 1 to 5."""
    prompt = f"""You are a strict evaluator.

Context: {context}

Question: {question}

Answer: {answer}

Is the answer fully supported by the context and does it answer the question?
Score 1 to 5 (5 = fully supported, 1 = not supported or wrong).
Reply with JSON only: {{"score": <1-5>, "reason": "<one sentence>"}}"""
    return ask_judge(prompt)

In [27]:
# A rough "is it really correct?" check using the gold answers,
# only used to see whether the judge agrees with the truth
def is_correct(answer, golds):
    answer_words = set(normalize(answer))
    for g in golds:
        gold_words = set(normalize(g))
        if gold_words and gold_words <= answer_words:
            return True
    return False

In [28]:
rows = []
for kind, answers in [("with passage", open_book_answers), ("without passage", closed_book_answers)]:
    for row, answer in zip(squad, answers):
        verdict = faithfulness(row["context"], row["question"], answer)
        rows.append({
            "answers given": kind,
            "answer": answer[:70],
            "really correct?": is_correct(answer, row["answers"]["text"]),
            "judge score": int(verdict["score"]),
        })

free_df = pd.DataFrame(rows)
free_df

,answers given,answer,really correct?,judge score
0,with passage,Edward Burne‑Jones,True,1
1,with passage,Slipback,True,5
2,with passage,Nafzger,True,5
3,with passage,Crash the Super Bowl,True,5
4,with passage,the oil shock,True,5
5,with passage,St. George's United Methodist Church,True,5
6,with passage,a hysterical and demonizing mentality about Jews,True,5
7,with passage,"the government, the National Assembly and the Senate",True,5
8,without passage,Louis Comfort Tiffany designed the ceiling and stained‑glass windows o,False,1
9,without passage,The first Doctor Who radio drama was “The Pescatons.”,False,1


In [29]:
# Does the judge agree with the truth? (score 4 or 5 = "supported")
agree = ((free_df["judge score"] >= 4) == free_df["really correct?"])
print(f"Judge agrees with the gold-answer check on {agree.sum()} of {len(free_df)} answers")

print("\nAverage judge score:")
print(free_df.groupby("answers given")["judge score"].mean())

summary.append(("Reference-free | judge agrees with gold check (out of %d)" % len(free_df), int(agree.sum())))

Judge agrees with the gold-answer check on 14 of 16 answers

Average judge score:
answers given
with passage       4.500
without passage    1.625
Name: judge score, dtype: float64


**What you're seeing:** the judge gave every answer a score **without ever seeing the gold answer**. Answers given with the passage should score higher than answers from memory, and the judge should mostly agree with our gold-answer check.

That is the promise of reference-free evaluation: **it scales with no ground truth**. The catch is in the rows where the judge disagreed. Read those rows: sometimes the judge is wrong (it can score an obviously correct answer low, or be fooled by odd input such as a blank answer, which is why `ask()` refuses blank answers), and sometimes our rough word-match check is wrong, because it counts any answer that contains the gold words, even one with extra unsupported claims. Nobody told the judge the right answer, so we need to know *when* to trust it. The rest of this notebook is about that.

| | Reference-free |
|---|---|
| **Advantages** | No ground truth needed. Scales to any number of cases. Handles open-ended answers. |
| **Disadvantages** | Harder to trust. The judge has its own mistakes and biases. Costs extra LLM calls. |

# 4. LLM as Judge: The Four Modes

A strong LLM reads the question, the context and the answer (generated by a weaker model), then scores it against criteria **you specify**. There are four ways to ask:

| Mode | What you ask | Best for |
|---|---|---|
| **Pointwise** | Score one answer on named dimensions | Absolute quality tracking |
| **Pairwise** | Which of two answers is better? | Comparing prompt or model versions |
| **Reference-guided** | How well does the answer capture the ground truth? | Paraphrase-tolerant correctness |
| **Criteria-based** | Check each named criterion separately | Structured, interpretable feedback |

We use the 6 Dolly questions. First we need a second set of answers: the same model with a **different prompt** (a detailed one).

In [30]:
# Second version of the answers: a detailed paragraph
detailed_answers = [ask(generator, q + "\n\nAnswer in one detailed paragraph of about 120 words.") for q in questions]

print("Concise  :", concise_answers[0])
print("\nDetailed :", detailed_answers[0])

Concise  : The standard tuning for a six‑string guitar is E‑A‑D‑G‑B‑E, from the lowest (thickest) string to the highest (thinnest). This arrangement is often remembered as “EADGBE.”

Detailed : The standard tuning for a six‑string guitar is E‑A‑D‑G‑B‑E, with the lowest‑pitched string (the “6th” string) tuned to the E two octaves below middle C (E₂, 82.4 Hz) and the highest‑pitched string (the “1st” string) tuned to the E an octave above middle C (E₄, 329.6 Hz). The strings are spaced in perfect fourths except for the interval between the G and B strings, which is a major third, giving the sequence of intervals E–A (perfect fourth), A–D (perfect fourth), D–G (perfect fourth), G–B (major third), and B–E (perfect fourth). This arrangement balances chordal harmony and scale patterns, making it the most widely adopted tuning for genres ranging from classical to rock, and it facilitates familiar fingerings, barre chords, and melodic runs across the fretboard.


## 4.1 Pointwise: score one answer on named dimensions

In [31]:
def pointwise(question, answer):
    """Score one answer on accuracy and clarity (1 to 5)."""
    prompt = f"""You are a strict evaluator.

Question: {question}

Answer: {answer}

Score the answer from 1 to 5 on each dimension:
- accuracy: are the facts correct?
- clarity: is it easy to understand?
Reply with JSON only: {{"accuracy": <1-5>, "clarity": <1-5>, "reason": "<one sentence>"}}"""
    return ask_judge(prompt)

rows = []
for q, answer in zip(questions, concise_answers):
    score = pointwise(q, answer)
    rows.append({"question": q[:60], "accuracy": score["accuracy"], "clarity": score["clarity"], "reason": score["reason"]})

pd.DataFrame(rows)

,question,accuracy,clarity,reason
0,What is the standard tuning for a 6 string guitar?,5,5,The answer correctly identifies the standard tuning notes in the proper order and prov...
1,What does a typical financial planning and analysis team str,5,5,"The answer accurately describes the standard hierarchical structure of an FP&A team, i..."
2,How should I pick a name for my baby?,5,5,"The advice is factually sound, covering both subjective preferences and practical cons..."
3,What is a jet engine?,5,5,The answer correctly defines a jet engine as a reaction engine and accurately describe...
4,How do I turn on my Tesla Y?,5,5,The answer correctly describes the standard procedure for starting a Tesla Model Y and...
5,Do you prefer a Tesla model S or a Porsche Taycan,5,5,The answer correctly identifies the key strengths of both vehicles (range/network vs. ...


**What you're seeing:** every answer gets its own scores on named dimensions, so you can **track quality over time**. No second answer or reference needed.

## 4.2 Pairwise: which of two answers is better?

Here we compare our two prompts: concise (A) vs detailed (B).

In [32]:
def pairwise(question, answer_a, answer_b, rubric=""):
    """Ask the judge which of two answers is better."""
    prompt = f"""You are a strict evaluator. Decide which answer is better.

Question: {question}

Answer A: {answer_a}

Answer B: {answer_b}

{rubric}
Reply with JSON only: {{"winner": "A" or "B", "reason": "<one sentence>"}}"""
    return ask_judge(prompt)

def first_wins(result):
    """True if the judge picked the answer shown FIRST (A)."""
    return str(result["winner"]).strip().strip(".").upper()[-1] == "A"

In [33]:
winners_order1 = []   # who won when concise = A and detailed = B
first_flags_order1 = []   # did the FIRST answer win?

for q, concise, detailed in zip(questions, concise_answers, detailed_answers):
    result = pairwise(q, concise, detailed)    # A = concise, B = detailed
    winners_order1.append("concise" if first_wins(result) else "detailed")
    first_flags_order1.append(first_wins(result))

print("Winners:", winners_order1)
print("Concise won   :", winners_order1.count("concise"), "of", len(questions))
print("Detailed won  :", winners_order1.count("detailed"), "of", len(questions))

Winners: ['detailed', 'detailed', 'detailed', 'detailed', 'detailed', 'detailed']
Concise won   : 0 of 6
Detailed won  : 6 of 6


**What you're seeing:** a head-to-head result between two **versions** of our prompt. This is how you decide "did my prompt change make things better?". Keep these results, because we test them for bias in Section 5.

## 4.3 Reference-guided: fixing Section 2's problem

Remember how F1 punished correct paraphrases on Dolly? Now we give the judge the **reference as a guide** and ask whether the answer captures its *meaning*.

In [34]:
def reference_guided(question, reference, answer):
    """Does the answer capture the meaning of the reference answer?"""
    prompt = f"""You are a strict evaluator.

Question: {question}

Reference answer (ground truth): {reference}

Model answer: {answer}

Does the model answer capture the meaning of the reference answer? The wording can differ.
Reply with JSON only: {{"verdict": "correct" or "partially correct" or "incorrect", "reason": "<one sentence>"}}"""
    return ask_judge(prompt)

rows = []
for q, ref, answer, f in zip(questions, references, concise_answers, dolly_df["F1 vs reference"]):
    result = reference_guided(q, ref, answer)
    rows.append({"question": q[:60], "F1 (word overlap)": f, "judge verdict": result["verdict"]})

guided_df = pd.DataFrame(rows)
guided_df

,question,F1 (word overlap),judge verdict
0,What is the standard tuning for a 6 string guitar?,0.54,correct
1,What does a typical financial planning and analysis team str,0.23,partially correct
2,How should I pick a name for my baby?,0.22,partially correct
3,What is a jet engine?,0.29,partially correct
4,How do I turn on my Tesla Y?,0.35,partially correct
5,Do you prefer a Tesla model S or a Porsche Taycan,0.26,partially correct


In [35]:
verdicts = guided_df["judge verdict"].str.lower()
fully = (verdicts == "correct").sum()
good = verdicts.isin(["correct", "partially correct"]).sum()
print(f"Judge says correct: {fully} of {len(guided_df)}")
print(f"Judge says correct or partially correct: {good} of {len(guided_df)}")
print("Average F1 on the same answers          :", round(guided_df["F1 (word overlap)"].mean(), 2))

summary.append(("Reference-guided judge | Dolly answers judged fully correct (out of %d)" % len(guided_df), int(fully)))
summary.append(("Reference-guided judge | Dolly answers judged correct or partially (out of %d)" % len(guided_df), int(good)))

Judge says correct: 1 of 6
Judge says correct or partially correct: 6 of 6
Average F1 on the same answers          : 0.32


**What you're seeing:** the **same answers** with a low F1 are now judged **correct or partially correct**, because the judge compares *meaning*, not words. Look at the verdict column: the judge is more tolerant of wording than F1, but it can still withhold full credit, so "partially correct" is not the same as "correct". Reference-guided judging gives you the paraphrase tolerance that plain string overlap lacks.

## 4.4 Criteria-based: check each criterion separately

Instead of one score, we get a **checklist**. We run it on the long, detailed answers because they are more likely to fail something.

In [36]:
def criteria_check(question, answer):
    """Check three criteria separately and give feedback."""
    prompt = f"""You are a strict evaluator.

Question: {question}

Answer: {answer}

Check each criterion separately:
- answers_question: does it directly answer the question?
- factually_accurate: are the facts correct?
- no_padding: is it free of filler and repetition?
Reply with JSON only: {{"answers_question": true or false, "factually_accurate": true or false, "no_padding": true or false, "feedback": "<one sentence on what to fix>"}}"""
    return ask_judge(prompt)

rows = []
for q, answer in zip(questions, detailed_answers):
    result = criteria_check(q, answer)
    rows.append({"question": q[:50], **result})

pd.DataFrame(rows)

,question,answers_question,factually_accurate,no_padding,feedback
0,What is the standard tuning for a 6 string guitar?,True,True,False,"The answer includes unnecessary details about frequencies, interval theory, and genre ..."
1,What does a typical financial planning and analysi,True,True,True,"The answer is concise, accurate, and directly addresses the question without unnecessa..."
2,How should I pick a name for my baby?,True,True,True,"The answer is concise, directly addresses the prompt with actionable advice, and conta..."
3,What is a jet engine?,True,True,True,"The answer is accurate, direct, and concise, requiring no changes."
4,How do I turn on my Tesla Y?,True,True,False,"Remove the irrelevant information about 'Walk-Away Lock' and remote start, as the ques..."
5,Do you prefer a Tesla model S or a Porsche Taycan,True,True,True,The response appropriately declines to express a personal preference while providing a...


**What you're seeing:** one row per answer with a **true/false per criterion** and a sentence of feedback. This is the most interpretable mode: you immediately see *what* failed, not just a number.

# 5. Judge Biases: Can We Trust the Judge?

A judge is an LLM, so it has quirks. From your notes (memorize these):

| Bias | What happens | Mitigation | Tested here? |
|---|---|---|---|
| **Position** | In pairwise, the first answer is favoured | Run both orders, keep consistent wins only | Yes (5.1) |
| **Verbosity** | Longer answers score higher even if worse | Penalize padding in the rubric, test short-vs-long pairs | Yes (5.2) |
| **Domain gaps** | The judge can't verify specialist facts | Use reference answers or expert review | Yes (5.3) |
| **Sycophancy** | Confident-sounding answers are believed | Don't anchor the judge, ask for evidence first | Seen in 5.3 |
| **Self-preference** | A judge favours its own model family's style | Judge with a different family than the generator | Applied, not tested |

> **Self-preference:** a fair test needs many samples from several model families, and a handful of examples would mislead. So we only **apply the mitigation**: our judge (Qwen) is a different family from our generator (gpt-oss).

## 5.1 Position bias: does the order matter?

We repeat every pairwise comparison with the **order swapped** (detailed = A, concise = B). A fair judge picks the same answer both times.

In [37]:
winners_order2 = []   # who won when detailed = A and concise = B
first_flags_order2 = []

for q, concise, detailed in zip(questions, concise_answers, detailed_answers):
    result = pairwise(q, detailed, concise)    # order swapped
    winners_order2.append("detailed" if first_wins(result) else "concise")
    first_flags_order2.append(first_wins(result))

position_df = pd.DataFrame({
    "question": [q[:60] for q in questions],
    "winner (concise first)": winners_order1,
    "winner (detailed first)": winners_order2,
})
position_df["consistent?"] = position_df["winner (concise first)"] == position_df["winner (detailed first)"]
position_df

,question,winner (concise first),winner (detailed first),consistent?
0,What is the standard tuning for a 6 string guitar?,detailed,detailed,True
1,What does a typical financial planning and analysis team str,detailed,detailed,True
2,How should I pick a name for my baby?,detailed,detailed,True
3,What is a jet engine?,detailed,detailed,True
4,How do I turn on my Tesla Y?,detailed,detailed,True
5,Do you prefer a Tesla model S or a Porsche Taycan,detailed,detailed,True


In [38]:
first_position_wins = sum(first_flags_order1) + sum(first_flags_order2)
flipped = int((~position_df["consistent?"]).sum())

print(f"The answer shown FIRST won {first_position_wins} of {2 * len(questions)} comparisons")
print(f"Verdict changed when we swapped the order: {flipped} of {len(questions)} questions")

# Mitigation: keep only the consistent wins
print("Consistent wins we can trust:", position_df[position_df["consistent?"]]["winner (concise first)"].tolist())

summary.append(("Position bias | pairs where the verdict flipped when order swapped (out of %d)" % len(questions), flipped))

The answer shown FIRST won 6 of 12 comparisons
Verdict changed when we swapped the order: 0 of 6 questions
Consistent wins we can trust: ['detailed', 'detailed', 'detailed', 'detailed', 'detailed', 'detailed']


**What you're seeing:** if the judge were fair, being first or second would not matter, and the winner would never flip. If some rows say `False` under `consistent?`, **the order changed the verdict**: that's position bias. The fix from your notes: run both orders and **only trust the consistent wins**.

*(With only 6 pairs it may show few or no flips. Absence of flips here doesn't prove the bias is absent, which is exactly why you keep running both orders.)*

## 5.2 Verbosity bias: does longer look better?

We take 4 concise answers and make a **padded version** of each: same facts, three times the words, with filler. Then the judge compares short vs padded, in both orders. The padded answer should never win, as it adds nothing.

In [39]:
q4 = questions[:4]
short4 = concise_answers[:4]

# Padded versions: same facts, just stretched with filler
padded4 = [
    ask(generator, "Rewrite this answer so it is about three times longer by adding filler phrases and repetition. "
                   "Do NOT add any new facts.\n\nAnswer: " + a)
    for a in short4
]

print("Short :", short4[0])
print("\nPadded:", padded4[0])

Short : The standard tuning for a six‑string guitar is E‑A‑D‑G‑B‑E, from the lowest (thickest) string to the highest (thinnest). This arrangement is often remembered as “EADGBE.”

Padded: The standard tuning for a six‑string guitar, that is to say the most common and widely‑used tuning that virtually every guitarist learns first, is E‑A‑D‑G‑B‑E. In other words, when you look at the strings from the lowest‑pitched, thickest string at the top of the instrument down to the highest‑pitched, thinnest string at the bottom, the notes you encounter, in order, are E, then A, then D, then G, then B, and finally another E. This very same arrangement is often, and very conveniently, remembered by the simple mnemonic “EADGBE,” which simply strings together the letters of each note in the exact order they appear.


In [40]:
def count_padded_wins(rubric=""):
    """How many of the 8 comparisons (4 pairs x 2 orders) did the padded answer win?"""
    padded_wins = 0
    for q, short, padded in zip(q4, short4, padded4):
        r1 = pairwise(q, short, padded, rubric)    # padded is B
        r2 = pairwise(q, padded, short, rubric)    # padded is A
        padded_wins += (not first_wins(r1)) + first_wins(r2)
    return padded_wins

no_rubric = count_padded_wins()
print("Padded answer won (plain prompt):", no_rubric, "of 8")

Padded answer won (plain prompt): 0 of 8


In [41]:
# Mitigation: a rubric that penalizes padding
anti_padding = ("Do not prefer an answer just because it is longer. Penalize filler and repetition. "
                "Prefer the answer that is correct and says it most efficiently.")

with_rubric = count_padded_wins(anti_padding)
print("Padded answer won (with anti-padding rubric):", with_rubric, "of 8")

summary.append(("Verbosity bias | padded answer won, plain prompt (out of 8)", no_rubric))
summary.append(("Verbosity bias | padded answer won, anti-padding rubric (out of 8)", with_rubric))

Padded answer won (with anti-padding rubric): 0 of 8


**What you're seeing:** two counts out of 8. Since the padded answer says nothing new, a fair judge should rarely pick it. If the first count is above zero, the judge was swayed by length alone. The second count shows whether telling the judge to **penalize padding** helped. If both counts are 0, this run shows no verbosity bias to fix, so it cannot show whether the rubric helps.

## 5.3 Domain gaps and sycophancy: the judge can't verify facts

Without the passage, some answers from Section 3 were **wrong, but sounded confident**. We take only the wrong ones and ask two kinds of judge:
1. **No reference:** "is this answer factually correct?" The judge has to rely on its own knowledge.
2. **Reference-guided:** the judge is given the gold answer.

In [42]:
def naive_check(question, answer):
    """No reference: the judge must rely on its own knowledge."""
    prompt = f"""Question: {question}

Answer: {answer}

Is the answer factually correct?
Reply with JSON only: {{"correct": true or false, "reason": "<one sentence>"}}"""
    return ask_judge(prompt)

# Pick the closed-book answers that really are wrong
wrong = [i for i, (row, a) in enumerate(zip(squad, closed_book_answers)) if a.strip() and not is_correct(a, row["answers"]["text"])]
print(f"Wrong answers found: {len(wrong)} of {len(squad)}")

Wrong answers found: 6 of 8


In [43]:
believed = 0       # wrong answers the no-reference judge accepted
caught = 0         # wrong answers the reference-guided judge rejected

for i in wrong:
    row, answer = squad[i], closed_book_answers[i]
    gold = row["answers"]["text"][0]

    naive = naive_check(row["question"], answer)
    guided = reference_guided(row["question"], gold, answer)

    believed += bool(naive["correct"])
    caught += (guided["verdict"].lower() == "incorrect")

    print("Q    :", row["question"])
    print("Gold :", gold)
    print("Model:", answer)
    print("No-reference judge says correct?:", naive["correct"], "| Reference-guided verdict:", guided["verdict"], "\n")

print(f"No-reference judge accepted {believed} of {len(wrong)} wrong answers")
print(f"Reference-guided judge rejected {caught} of {len(wrong)} wrong answers")

summary.append(("Domain gap | wrong answers accepted by judge with no reference (out of %d)" % len(wrong), believed))
summary.append(("Domain gap | wrong answers rejected by judge with the reference (out of %d)" % len(wrong), caught))

Q    : Who designed the ceiling and stained-glass windows of the Green Dining Room?
Gold : Edward Burne-Jones
Model: Louis Comfort Tiffany designed the ceiling and stained‑glass windows of the Green Dining Room.
No-reference judge says correct?: True | Reference-guided verdict: incorrect 

Q    : What was the name of the first Doctor Who radio drama?
Gold : Slipback
Model: The first Doctor Who radio drama was “The Pescatons.”
No-reference judge says correct?: False | Reference-guided verdict: incorrect 

Q    : Who compiled the original surviving Apollo 11 landing data?
Gold : Nafzger
Model: NASA’s Manned Spacecraft Center (now Johnson Space Center) compiled the original surviving Apollo 11 landing data.
No-reference judge says correct?: True | Reference-guided verdict: incorrect 

Q    : What church is located at the corner of 4th and New Streets in Philadelphia?
Gold : St. George's United Methodist Church
Model: St. John the Baptist Catholic Church is located at the corner of 4th and

**What you're seeing:** for each wrong answer, you can compare what the judge said **with** and **without** the answer key. Our "wrong" list comes from a rough word-matching check, so a few of these answers may actually be right but worded differently. Read the printed rows and use your own judgement.

Without a reference, the judge can only use its own memory. If the answer **sounds confident** and the fact is specific (here, details from one Wikipedia passage), it can be believed. With the reference, the judge can **check instead of trust**.

This is why your notes say: *use reference answers or expert review* for domain gaps, and *ask for evidence first* so the judge isn't anchored by confident wording.

# 6. Final Comparison

## 6.1 The real numbers from this notebook

In [44]:
pd.DataFrame(summary, columns=["experiment", "result"])

,experiment,result
0,Reference-based | SQuAD (one right answer) | average F1,0.99
1,Reference-based | Dolly (open-ended) | average F1,0.32
2,Reference-free | judge agrees with gold check (out of 16),14.00
3,Reference-guided judge | Dolly answers judged fully correct (out of 6),1.00
4,Reference-guided judge | Dolly answers judged correct or partially (out of 6),6.00
5,Position bias | pairs where the verdict flipped when order swapped (out of 6),0.00
6,"Verbosity bias | padded answer won, plain prompt (out of 8)",0.00
7,"Verbosity bias | padded answer won, anti-padding rubric (out of 8)",0.00
8,Domain gap | wrong answers accepted by judge with no reference (out of 6),4.00
9,Domain gap | wrong answers rejected by judge with the reference (out of 6),6.00


## 6.2 Which one should you use?

| Approach | Advantages | Disadvantages | Use when |
|---|---|---|---|
| **Reference-based** (EM, F1) | Cheap, objective, repeatable | Needs ground truth for every case. Punishes valid paraphrases | One right answer: extraction, classification, multiple choice |
| **Reference-free** (LLM judge) | No ground truth. Scales. Handles open-ended answers | Harder to trust. Extra LLM calls. Judge biases | Open-ended generation, RAG faithfulness, large volumes |
| **Pointwise** | Absolute scores to track over time | Scores can drift and be inconsistent | Monitoring quality |
| **Pairwise** | Easier for a judge than absolute scores | Position bias, needs two answers | Comparing prompt or model versions |
| **Reference-guided** | Paraphrase-tolerant correctness. Fixes domain gaps | Needs a reference answer | Correctness when you do have ground truth |
| **Criteria-based** | Interpretable, tells you *what* failed | More criteria means more calls and more to design | Structured feedback and debugging |

## 6.3 Key takeaways

1. Normal `assert` tests fail on LLM output because **many different answers are correct**.
2. **Reference-based** scoring is great when there is one right answer and weak for open-ended text.
3. **Reference-free** scoring scales, and an **LLM judge** is the usual way to do it.
4. Pick the **judge mode** by the question you are asking: absolute quality, A vs B, correctness, or a checklist.
5. A judge has biases. **Run both orders, penalize padding, give it references, and use a different model family.**
6. Always check a judge against something you trust before relying on it.